# Домашнее задание 3 — Агент-аналитик системных промптов (MCP + Skills)

### Задания

1. **Файловые инструменты агента** — реализовать инструменты для сохранения/чтения результатов анализа
2. **Пайплайн: агенты с tool loop** — реализовать агента и оркестратор пайплайна

### Критерии оценки (10 баллов)

| # | Критерий | Балл |
|---|----------|------|
| 1 | Реализованы все 5 файловых инструментов (`save_analysis`, `list_analyses`, `read_analysis`, `save_report`, `read_report`) + tool schemas + `LOCAL_TOOLS` | 1 |
| 2 | Написаны все 4 skill-файла (`skills.md`, `prompt_analysis.md`, `analysis_report.md`, `prompt_generator.md`) с двухуровневой структурой (Level 1 — метаданные в `skills.md`, Level 2 — полная процедура) | 1 |
| 3 | `prompt_analysis.md` содержит ≥5 критериев оценки промпта (Role/Persona, Few-shot, Chain-of-thought, Output format, Guardrails и т.д.) | 1 |
| 4 | Реализован `run_agent` с корректным tool loop: маршрутизация вызовов (MCP / `read_skill` / локальные), трассировка через `AgentTrace`, чистый контекст на каждый вызов | 1 |
| 5 | Сценарий `prompt_analysis` работает: агент загружает skill → читает файл из GitHub → анализирует → сохраняет через `save_analysis` | 1 |
| 6 | Сценарий `analysis_report` работает: агент читает все анализы → формирует сводный отчёт → сохраняет через `save_report` | 1 |
| 7 | Сценарий `prompt_generator` работает: агент читает отчёт → генерирует новый системный промпт → сохраняет результат | 1 |
| 8 | Progressive disclosure реализован как для skills, так и для MCP-инструментов | 1 |
| 9–10 | Сценарий `prompt_analysis` реализован через субагента (отдельный агентный вызов для каждого файла, а не один монолитный контекст) | 2 |

---
## 0. Подготовка

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import sys
import json
import asyncio
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import nest_asyncio

nest_asyncio.apply()

from src.config import settings
from openai import OpenAI

In [ ]:
# ============================================================
# Настройка LLM-клиентов
# ============================================================

client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)

MODEL = "openai/gpt-4o"

---
## 0.1 Утилиты для трассировки агента

Код из практики — необходим для выполнения заданий.

In [ ]:
# ============================================================
# Утилиты для трассировки агента (из практики 3)
# ============================================================
from dataclasses import dataclass, field


def _extract_usage(response) -> dict:
    """Извлечь токены из response.usage."""
    usage = getattr(response, "usage", None)
    if not usage:
        return {"prompt_tokens": 0, "cached_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "context_size": 0}

    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or 0

    cached_tokens = getattr(usage, "prompt_cache_hit_tokens", 0) or 0
    if not cached_tokens:
        details = getattr(usage, "prompt_tokens_details", None)
        if details:
            cached_tokens = getattr(details, "cached_tokens", 0) or 0

    context_size = prompt_tokens + cached_tokens
    return {
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cached_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "context_size": context_size,
    }


@dataclass
class AgentStep:
    """Один шаг агента — для трассировки."""

    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observations: list[dict] = field(default_factory=list)
    prompt_tokens: int = 0
    cached_tokens: int = 0
    context_size: int = 0
    context_size_delta: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    final_answer: str | None = None


@dataclass
class AgentTrace:
    """Полный лог работы агента."""

    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    prompt_tokens_total: int = 0
    completion_tokens_total: int = 0
    total_tokens_total: int = 0
    latest_context_size: int = 0
    max_context_size: int = 0
    final_answer: str = ""

    def summary(self) -> str:
        lines = [
            f"Вопрос: {self.question}",
            f"Шагов: {self.total_steps}",
            f"Prompt tokens (биллинг): {self.prompt_tokens_total}",
            f"Completion tokens: {self.completion_tokens_total}",
            f"Всего tokens: {self.total_tokens_total}",
            "",
        ]
        for s in self.steps:
            lines.append(f"--- Шаг {s.step_number} ---")
            if s.total_tokens:
                lines.append(
                    f"  \U0001f522 context={s.context_size} (\u0394 {s.context_size_delta:+}), "
                    f"completion={s.completion_tokens}"
                )
            for tc in s.tool_calls:
                lines.append(f"  \U0001f527 {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)[:80]})")
            for obs in s.observations:
                text = str(obs)
                lines.append(f"  \U0001f4e1 {text[:120]}{'...' if len(text) > 120 else ''}")
            if s.final_answer:
                lines.append(f"  \U0001f4ac {s.final_answer[:300]}")
        lines.append(f"\n{'=' * 50}\n\U0001f3af Финальный ответ:\n{self.final_answer[:800]}")
        return "\n".join(lines)

---
## 0.2 Источник данных — GitHub MCP Server

Подключаемся к официальному GitHub MCP-серверу для доступа к репозиторию с системными промптами.

Репозиторий [`asgeirtj/system_prompts_leaks`](https://github.com/asgeirtj/system_prompts_leaks) — коллекция реальных системных промптов (ChatGPT, Claude, Gemini, Codex и др.)

### Предварительная настройка

#### 1. Node.js и npx

**npx** — утилита из экосистемы Node.js для запуска npm-пакетов без установки.

Проверить установку:
```bash
node --version   # должно быть v18+ 
npx --version    # должно быть 9+
```

#### 2. GitHub Personal Access Token

Создаётся в [Settings → Developer settings → Personal access tokens → Tokens (classic)](https://github.com/settings/tokens).

Минимальные права: `public_repo`.

Добавьте токен в файл `.env` в корне проекта:
```
GITHUB_TOKEN=ghp_ваш_токен_здесь
```

In [ ]:
# ============================================================
# Подключение к GitHub MCP Server через stdio
# ============================================================
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Параметры подключения — запускаем официальный GitHub MCP-сервер
github_server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-github"],
    env={
        **os.environ,
        "GITHUB_PERSONAL_ACCESS_TOKEN": settings.github_token,
    },
)

In [ ]:
# ============================================================
# Discovery: подключаемся к GitHub MCP и узнаём его возможности
# ============================================================


async def discover_github_server():
    """Подключиться к GitHub MCP-серверу и получить список инструментов."""
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("\u2705 Подключение к GitHub MCP установлено!\n")

                # Discovery: Tools
                tools_response = await session.list_tools()
                print(f"\U0001f527 Инструменты ({len(tools_response.tools)}):")
                print("-" * 55)
                for tool in tools_response.tools:
                    desc = (tool.description or "")[:80]
                    print(f"  \u2022 {tool.name}")
                    print(f"    {desc}...")
                    print()

                return tools_response.tools


github_tools = asyncio.get_event_loop().run_until_complete(discover_github_server())

---
## 0.3 Skills — процедурные навыки агента

В практике мы создали skills для космического ассистента.  
Здесь используем **три навыка для анализа промптов**:

| Навык | Файл | Назначение |
|-------|------|------------|
| `prompt_analysis` | `hw_3_skills/prompt_analysis.md` | Пошаговый разбор системного промпта: секции, persona, приёмы, оценки |
| `analysis_report` | `hw_3_skills/analysis_report.md` | Формирование сводного отчёта с таблицами, цитатами и рекомендациями |
| `prompt_generator` | `hw_3_skills/prompt_generator.md` | Генерация нового промпта на основе выявленных паттернов |

### Skill Chaining

В практике skills были **независимые**. Здесь они образуют **цепочку** (chain):

```
prompt_analysis ──(структурированный разбор)──→ analysis_report
                                                       │
                                              (отчёт + рекомендации)
                                                       │
                                                       ▼
                                               prompt_generator
```

**Выход одного skill = вход следующего.** Агент сам решает, когда переключиться.

In [ ]:
# ============================================================
# Реестр Skills — загружаем все skill-файлы из hw_3_skills/
# ============================================================

from pydantic import BaseModel, Field


class SkillInfo(BaseModel):
    """Метаданные одного skill-файла."""

    name: str = Field(description="имя файла (без .md)")
    title: str = Field(description="Заголовок skill")
    trigger_description: str = Field(description="Когда активировать (из секции 'Когда активировать')")
    content: str = Field(description="Полный текст skill")


def load_skills(skills_directory: Path) -> dict[str, SkillInfo]:
    """Загрузить все skill-файлы из директории (кроме skills.md)."""
    skills = {}

    for skill_file in sorted(skills_directory.glob("*.md")):
        if skill_file.name == "skills.md":
            continue

        with open(skill_file, "r", encoding="utf-8") as f:
            content = f.read()

        lines = content.split("\n")
        title = lines[0].replace("# Skill: ", "").strip() if lines else skill_file.stem

        trigger = ""
        in_trigger = False
        for line in lines:
            if "Когда активировать" in line:
                in_trigger = True
                continue
            if in_trigger:
                if line.startswith("##") and "Когда" not in line:
                    break
                trigger += line + "\n"

        name = skill_file.stem
        skills[name] = SkillInfo(
            name=name,
            title=title,
            trigger_description=trigger.strip(),
            content=content,
        )

    return skills


def load_skills_metadata(skills_directory: Path) -> str:
    """Загрузить skills.md — Level 1 метаданные для system prompt."""
    skills_md = skills_directory / "skills.md"
    if skills_md.exists():
        with open(skills_md, "r", encoding="utf-8") as f:
            return f.read()
    return "\u26a0\ufe0f Файл skills.md не найден в директории skills."


# --- Загрузка ---
skills_dir = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_skills_solved"
SKILLS = load_skills(skills_dir)

print(f"\U0001f4c2 Директория skills: {skills_dir}")
print(f"\U0001f4cb Загружено skills: {len(SKILLS)}\n")

for name, info in SKILLS.items():
    print(f"  \u2022 {name}: {info.title}")
    print(f"    Триггер: {info.trigger_description[:80]}...")
    print()

📂 Директория skills: c:\Users\isami\OneDrive\Рабочий стол\Projects\Personal\ВШЭ\HSE-Agent-Systems\lectures\lecture_3\hw_3_skills_solved
📋 Загружено skills: 3

  • analysis_report: Формирование аналитического отчёта
    Триггер: Этот skill активируется, когда:
- Анализ промпта (навык `prompt_analysis`) завер...

  • prompt_analysis: Анализ системного промпта
    Триггер: Этот skill активируется, когда нужно:
- Разобрать системный промпт на составные ...

  • prompt_generator: Генерация системного промпта
    Триггер: Этот skill активируется, когда:
- Аналитический отчёт (навык `analysis_report`) ...



In [ ]:
# ============================================================
# Level 1: загружаем skills.md — краткий обзор навыков для system prompt
# ============================================================

skills_metadata = load_skills_metadata(skills_dir)
print("Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):\n")
print(skills_metadata)

Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):

# Skills — Навыки агента-аналитика системных промптов

Набор skills, определяющих **как** агент должен действовать при анализе системных промптов, формировании отчётов и генерации новых промптов.

## Список навыков

| Навык | Файл | Краткое описание |
|-------|------|------------------|
| Анализ промпта | [prompt_analysis.md](prompt_analysis.md) | Разбор системного промпта на составные части: секции, persona, приёмы prompt engineering, оценка качества. Результат сохраняется через `save_analysis()` |
| Аналитический отчёт | [analysis_report.md](analysis_report.md) | Агрегация результатов анализа нескольких промптов: матрицы секций и приёмов, общие паттерны, лучшие практики, сравнительные оценки. Результат сохраняется через `save_report()` |
| Генерация промпта | [prompt_generator.md](prompt_generator.md) | Создание нового системного промпта на основе выявленных лучших практик из аналитического отчёта. Результат со

In [ ]:
# ============================================================
# Инструмент read_skill — загрузка процедуры по требованию
# ============================================================

READ_SKILL_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_skill",
        "description": (
            "Загрузить детальную процедуру (skill) для решения задачи определённого типа. "
            "Вызывай этот инструмент, когда задача пользователя соответствует одному из "
            "доступных skills. После загрузки — строго следуй процедуре шаг за шагом."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": ("Имя skill для загрузки: " + ", ".join(SKILLS.keys())),
                }
            },
            "required": ["name"],
        },
    },
}


def read_skill(name: str) -> str:
    """Загрузить полный контент skill по имени."""
    if name in SKILLS:
        skill = SKILLS[name]
        return (
            f"\u2550\u2550\u2550 SKILL ЗАГРУЖЕН: {skill.title} \u2550\u2550\u2550\n\n"
            f"СТРОГО следуй этой процедуре шаг за шагом.\n\n"
            f"{skill.content}"
        )
    available = ", ".join(SKILLS.keys())
    return f"Skill '{name}' не найден. Доступные skills: {available}"


# --- Тест ---
print("\U0001f527 READ_SKILL_TOOL_SCHEMA:")
print(json.dumps(READ_SKILL_TOOL_SCHEMA, ensure_ascii=False, indent=2)[:500])
print("\n--- Тест: read_skill('prompt_analysis') ---")
print(read_skill("prompt_analysis")[:400] + "...")

🔧 READ_SKILL_TOOL_SCHEMA:
{
  "type": "function",
  "function": {
    "name": "read_skill",
    "description": "Загрузить детальную процедуру (skill) для решения задачи определённого типа. Вызывай этот инструмент, когда задача пользователя соответствует одному из доступных skills. После загрузки — строго следуй процедуре шаг за шагом.",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "Имя skill для загрузки: analysis_report, prompt_a

--- Тест: read_skill('prompt_analysis') ---
═══ SKILL ЗАГРУЖЕН: Анализ системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Анализ системного промпта

## Когда активировать
Этот skill активируется, когда нужно:
- Разобрать системный промпт на составные части
- Понять структуру и приёмы, использованные в промпте
- Выявить persona, constraints, инструкции и форматирование
- Оценить качество и полноту промпта

## Проток...


---
## 0.4 Вспомогательные функции пайплайна

Код из практики — утилиты для обхода репозитория и конвертации MCP-инструментов.

In [ ]:

def format_mcp_tools(mcp_tools) -> list[dict]:
    """Конвертировать MCP tools в OpenAI function-calling формат."""
    formatted = []
    for tool in mcp_tools:
        schema = tool.inputSchema if tool.inputSchema else {"type": "object", "properties": {}}
        formatted.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": schema,
                },
            }
        )
    return formatted


---
## Задание 1: Файловые инструменты агента

Агент работает в несколько этапов, и между этапами ему нужно **сохранять и читать результаты** через файловую систему.

### Что нужно реализовать

| Инструмент | Назначение | Когда используется |
|------------|-----------|--------------------|
| `save_analysis(name, content)` | Сохранить анализ одного промпта в файл | После анализа каждого промпта |
| `list_analyses()` | Получить список всех сохранённых анализов | Перед формированием отчёта |
| `read_analysis(name)` | Прочитать конкретный анализ | При формировании отчёта |
| `save_report(name, content)` | Сохранить сводный отчёт / промпт | После формирования отчёта / промпта |
| `read_report(name)` | Прочитать сохранённый отчёт | Перед генерацией промпта |

### Файловая структура

```
hw_3_output/
  analyses/       ← save_analysis / read_analysis / list_analyses
    OpenAI_Codex.md
    Anthropic_Claude.md
    ...
  reports/         ← save_report / read_report
    analysis_report.md
    generated_prompt.md
```

### Подсказки
- Не забудьте добавить tool schemas в формате OpenAI function-calling
- Соберите все локальные инструменты в маппинг `LOCAL_TOOLS`

In [ ]:
# ============================================================
# Инструменты агента
# ============================================================

# --- Директории для результатов ---
ANALYSES_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "analyses"
REPORTS_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "reports"

ANALYSES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\U0001f4c2 Директория анализов: {ANALYSES_DIR}")
print(f"\U0001f4c2 Директория отчётов:  {REPORTS_DIR}")


# ---- save_analysis ----
def save_analysis(name: str, content: str) -> str:
    """Сохранить сводку по анализу одного промпта."""
    # TODO: сохранить content в файл ANALYSES_DIR / f"{name}.md"
    # Вернуть строку: "✅ Анализ сохранён: {path.name} ({len(content)} символов)"
    ...


# ---- list_analyses ----
def list_analyses() -> str:
    """Получить список всех сохранённых анализов."""
    # TODO: получить список .md файлов из ANALYSES_DIR
    # Если пусто — вернуть "❌ Нет сохранённых анализов..."
    # Иначе — вернуть список с названиями и размерами
    ...


# ---- read_analysis ----
def read_analysis(name: str) -> str:
    """Прочитать сохранённый анализ промпта по имени."""
    # TODO: прочитать файл ANALYSES_DIR / f"{name}.md"
    # Если файл есть — вернуть "═══ АНАЛИЗ: {name} ═══\n\n{content}"
    # Если нет — вернуть список доступных
    ...


# ---- save_report ----
def save_report(name: str, content: str) -> str:
    """Сохранить отчёт или сгенерированный промпт."""
    # TODO: сохранить content в файл REPORTS_DIR / f"{name}.md"
    # Вернуть строку подтверждения
    ...


# ---- read_report ----
def read_report(name: str) -> str:
    """Прочитать сохранённый отчёт."""
    # TODO: прочитать файл REPORTS_DIR / f"{name}.md"
    # Если файл есть — вернуть содержимое
    # Если нет — вернуть список доступных
    ...


# ---- List files in github repo ----
async def list_repo_files(session) -> list[dict]:
    """
    Обойти репозиторий через MCP и собрать список промпт-файлов.
    Возвращает [{"path": "OpenAI/Codex.md", "name": "Codex.md", "folder": "OpenAI", "size": ...}]
    """
    # Параметры репозитория
    # REPO = {"owner": "asgeirtj", "repo": "system_prompts_leaks"}
    ...


In [ ]:
# Опишите каждый инструмент в формате OpenAI function-calling.

FILE_TOOL_SCHEMAS = [
    # TODO: опишите save_analysis(name, content)
    # TODO: опишите list_analyses() (без параметров)
    # TODO: опишите read_analysis(name)
    # TODO: опишите save_report(name, content)
    # TODO: опишите read_report(name)
]


# ============================================================
# Маппинг локальных инструментов
# ============================================================

LOCAL_TOOLS = {
    "read_skill": read_skill,
    # TODO: добавьте все файловые инструменты:
    # "save_analysis": save_analysis,
    # "list_analyses": list_analyses,
    # "read_analysis": read_analysis,
    # "save_report": save_report,
    # "read_report": read_report,
}

print(f"\n\U0001f527 Локальные инструменты агента ({len(LOCAL_TOOLS)}):")
for name in LOCAL_TOOLS:
    print(f"  \u2022 {name}")

---
## Пайплайн — универсальный агент с tool loop

Реализуйте **одного универсального агента** `run_agent()`, который выполняет разные задачи в зависимости от запроса (`Начни анализ`, `сформируй сводку`, `сгенерируй промпт для такой то задачи`). 
### Подсказки

- За основу tool loop возьмите `run_reactive_agent` из практики 2 (или аналог из практики 3)
- Используйте `AgentTrace` для отслеживания шагов и токенов

In [ ]:
# ============================================================
# Универсальный агент с tool loop
# ============================================================

# ⚠️ Этап анализа всех промптов может быть долгим, т.к. для каждого файла
# выполняется отдельный агентный вызов с несколькими шагами tool loop.
# Для ускорения вместо синхронных последовательных запросов можно создать
# N потоков (asyncio.gather / ThreadPoolExecutor) и запустить анализ
# нескольких промптов параллельно.
# Как это реализовать — идете на поклон к @Rinaaaa_chan


async def run_agent(
    # + остальные требуемые аргументы функции
    max_steps: int = 10,
    verbose: bool = True,
) -> AgentTrace:
    ...

In [ ]:
# ============================================================
# Тест кейс 1. Запрос на начало анализа 
# ============================================================


In [ ]:
# ============================================================
# Тест кейс 2. Запрос на формирование итоговой сводки
# ============================================================

In [ ]:
# ============================================================
# Тест кейс 2. Запрос на формирование промпта для случайного ассистента
# ============================================================

<img src="pictures\hw_file_1.PNG" width="600"/>

<img src="pictures\hw_file_2.PNG" width="600"/>